In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, median_absolute_error

#load in the cleaned test and training datasets
test_df = pd.read_csv("cleaned_test.csv")
train_df = pd.read_csv("cleaned_training.csv")

feature_cols = [
    'LivingArea', 
    'BedroomsTotal', 
    'BathroomsTotalInteger',
    'LotSizeSquareFeet', 
    'zip_median_price', 
    'city_median_price',
    'bed_bath_ratio', 
    'property_age', 
    'district_median_price'
]

X_train = train_df[feature_cols].copy()
y_train = train_df['ClosePrice'].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df['ClosePrice'].copy()

print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape}  | y_test shape:  {y_test.shape}")

X_train shape: (71099, 9) | y_train shape: (71099,)
X_test shape:  (12784, 9)  | y_test shape:  (12784,)


Train the XGB Model and save to joblib

In [12]:
import joblib

xgb_model = XGBRegressor(
    max_depth=9,
    learning_rate=0.1,
    n_estimators=300,
    random_state=42
)

xgb_model.fit(X_train, y_train)

print("XGBoost model trained successfully.")

joblib.dump(xgb_model, "xgb_model.pkl")

print("Saved xgb_model.pkl")

XGBoost model trained successfully.
Saved xgb_model.pkl


Create Raw Dataset that is not fully cleaned as done in previously saved datasets

In [13]:
from pathlib import Path
import geopandas as gpd

data_dir = Path.cwd().parent / "Data"

training_files = [
    data_dir / 'CRMLSSold202511.csv',
    data_dir / 'CRMLSSold202512.csv',
    data_dir / 'CRMLSSold202601.csv',
    data_dir / 'CRMLSSold202602.csv',
    data_dir / 'CRMLSSold202603.csv',
    data_dir / 'CRMLSSold202604.csv',
    data_dir / 'CRMLSSold202605.csv',
]

raw_dfs = [
    pd.read_csv(file, low_memory=False)
    for file in training_files
]

raw_df = pd.concat(raw_dfs, ignore_index=True)

# apply filters to keep only residential single-family homes with valid close prices
raw_df = raw_df[
    (raw_df["PropertyType"] == "Residential") &
    (raw_df["PropertySubType"] == "SingleFamilyResidence")
].copy()

raw_df["ClosePrice"] = pd.to_numeric(raw_df["ClosePrice"],errors="coerce")

raw_df = raw_df[
    raw_df["ClosePrice"].between(50_000, 10_000_000)
].copy()


# Clean ZIP codes and city names
# Clean ZIP codes
raw_df["PostalCode5"] = (
    raw_df["PostalCode"]
    .astype(str)
    .str.extract(r"(\d{5})", expand=False)
)

# Keep only valid 5-digit ZIP codes
raw_df = raw_df[raw_df["PostalCode5"].notna()].copy()


raw_df["City"] = (
    raw_df["City"]
    .astype(str)
    .str.strip()
    .replace({"": "Unknown", "nan": "Unknown"})
)

print(f"Cleaned rows: {len(raw_df):,}")
print(f"Unique ZIPs: {raw_df['PostalCode5'].nunique():,}")
print(f"Unique Cities: {raw_df['City'].nunique():,}")

# # Check for date-like and square footage columns
# date_like_cols = [c for c in raw_df.columns if 'date' in c.lower() or 'close' in c.lower()]
# sqft_like_cols = [c for c in raw_df.columns if 'area' in c.lower() or 'sqft' in c.lower() or 'sq_ft' in c.lower()]

# print("Date-like columns:", date_like_cols)
# print("Square-footage-like columns:", sqft_like_cols)
# print()
# print(raw_df[date_like_cols + sqft_like_cols].head())

Cleaned rows: 71,150
Unique ZIPs: 1,337
Unique Cities: 927


Create the geographical layer as done in week 6

In [14]:
# Load school district GeoJSON
data_dir = Path.cwd().parent / "Data"

school_file = data_dir / "school_districts.geojson"
school_gdf = gpd.read_file(school_file)

# Keep only Unified districts
unified_districts = school_gdf[school_gdf["DistrictType"] == "Unified"].copy()

# Convert properties into geographic points
train_gdf = gpd.GeoDataFrame(
    raw_df,
    geometry=gpd.points_from_xy(
        raw_df["Longitude"],
        raw_df["Latitude"]
    ),
    crs="EPSG:4326"
)

# Match coordinate systems
unified_districts = unified_districts.to_crs(train_gdf.crs)

# Spatial join
train_gdf = gpd.sjoin(
    train_gdf,
    unified_districts,
    how="left",
    predicate="within"
)

# Create SchoolDistrict
train_gdf["SchoolDistrict"] = (
    train_gdf["DistrictName"]
    .fillna("Unknown")
)

print("Top 10 School Districts represented:")
print(train_gdf["SchoolDistrict"].value_counts().head(10))

Top 10 School Districts represented:
SchoolDistrict
Unknown                 17224
Los Angeles Unified      7065
San Diego Unified        1966
Desert Sands Unified     1443
Capistrano Unified       1341
Palm Springs Unified     1255
Oakland Unified           931
Hemet Unified             885
Corona-Norco Unified      874
Long Beach Unified        870
Name: count, dtype: int64


In [15]:
# Global fallback
global_median = float(train_gdf["ClosePrice"].median())

# ZIP median
zip_median_lookup = (
    train_gdf
    .groupby("PostalCode5")["ClosePrice"]
    .median()
    .to_dict()
)

# City median
city_median_lookup = (
    train_gdf
    .groupby("City")["ClosePrice"]
    .median()
    .to_dict()
)

known_districts = train_gdf[train_gdf["SchoolDistrict"] != "Unknown"]

# School district median
district_median_lookup = (
    known_districts
    .groupby("SchoolDistrict")["ClosePrice"]
    .median()
    .to_dict()
)

print(f"Global median: ${global_median:,.0f}")
print(f"ZIPs: {len(zip_median_lookup):,}")
print(f"Cities: {len(city_median_lookup):,}")
print(f"Districts: {len(district_median_lookup):,}")

Global median: $885,000
ZIPs: 1,337
Cities: 927
Districts: 310


In [16]:
# Helper to select the mode while ignoring 'Unknown'
def most_common_known(values):
    clean_vals = values[values != "Unknown"]
    if clean_vals.empty:
        return "Unknown"
    return clean_vals.mode().iloc[0]

# Most common city for each ZIP
zip_to_city = (
    train_gdf
    .groupby("PostalCode5")["City"]
    .agg(most_common_known)
    .to_dict()
)


# Most common school district for each ZIP
zip_to_district = (
    train_gdf
    .groupby("PostalCode5")["SchoolDistrict"]
    .agg(most_common_known)
    .to_dict()
)

print("Sample ZIP -> City mapping:", list(zip_to_city.items())[:5])
print("Sample ZIP -> District mapping:", list(zip_to_district.items())[:5])

Sample ZIP -> City mapping: [('06021', 'Corning'), ('20536', 'Ontario'), ('22785', 'Other'), ('22880', 'Other'), ('39534', 'Lancaster')]
Sample ZIP -> District mapping: [('06021', 'Unknown'), ('20536', 'Unknown'), ('22785', 'Unknown'), ('22880', 'Unknown'), ('39534', 'Unknown')]


In [17]:
joblib.dump(zip_median_lookup, "zip_median_lookup.pkl")
joblib.dump(city_median_lookup, "city_median_lookup.pkl")
joblib.dump(district_median_lookup, "district_median_lookup.pkl")

joblib.dump(zip_to_city, "zip_to_city.pkl")
joblib.dump(zip_to_district, "zip_to_district.pkl")

joblib.dump(global_median, "global_median.pkl")

print("All lookup files exported successfully.")

All lookup files exported successfully.


In [18]:
# ==============================================================================
# EXPORT MARKET DATA FOR STREAMLIT MARKET TAB
# ==============================================================================

raw_df["CloseDate"] = pd.to_datetime(raw_df["CloseDate"], errors="coerce")
raw_df["ListingContractDate"] = pd.to_datetime(raw_df["ListingContractDate"], errors="coerce")
raw_df["LivingArea"] = pd.to_numeric(raw_df["LivingArea"], errors="coerce")
raw_df["Latitude"] = pd.to_numeric(raw_df["Latitude"], errors="coerce")
raw_df["Longitude"] = pd.to_numeric(raw_df["Longitude"], errors="coerce")

# Price per sqft — guard against missing/zero living area
market_df = raw_df[(raw_df["LivingArea"].notna()) & (raw_df["LivingArea"] > 0)].copy()
market_df["PricePerSqFt"] = market_df["ClosePrice"] / market_df["LivingArea"]

# Days on market, with a sanity filter to drop bad data entry (negative or multi-year outliers)
market_df["DaysOnMarket"] = (market_df["CloseDate"] - market_df["ListingContractDate"]).dt.days
market_df = market_df[(market_df["DaysOnMarket"] >= 0) & (market_df["DaysOnMarket"] <= 730)].copy()

# ---------- ZIP-level stats (for the map) ----------
zip_stats = (
    market_df.groupby("PostalCode5")
    .agg(
        median_price=("ClosePrice", "median"),
        median_ppsf=("PricePerSqFt", "median"),
        sales_count=("ClosePrice", "count"),
        latitude=("Latitude", "mean"),
        longitude=("Longitude", "mean"),
    )
    .reset_index()
)
zip_stats = zip_stats[zip_stats["sales_count"] >= 5]          # match reference's "min 5 sales" rule
zip_stats = zip_stats.dropna(subset=["latitude", "longitude"])

# ---------- City-level stats (for the bar chart) ----------
city_stats = (
    market_df.groupby("City")
    .agg(
        median_price=("ClosePrice", "median"),
        median_ppsf=("PricePerSqFt", "median"),
        sales_count=("ClosePrice", "count"),
        median_dom=("DaysOnMarket", "median"),
    )
    .reset_index()
    .sort_values("sales_count", ascending=False)
)

# ---------- Overall summary (for the top metric cards) ----------
market_summary = {
    "homes_sold": int(len(market_df)),
    "median_sale_price": float(market_df["ClosePrice"].median()),
    "median_ppsf": float(market_df["PricePerSqFt"].median()),
    "median_dom": float(market_df["DaysOnMarket"].median()),
}

# ---------- Price distribution (top/bottom 1% trimmed, matching reference's approach) ----------
lo_p, hi_p = market_df["ClosePrice"].quantile([0.01, 0.99])
lo_s, hi_s = market_df["PricePerSqFt"].quantile([0.01, 0.99])
price_distribution = {
    "close_price": market_df["ClosePrice"].clip(lo_p, hi_p).round(0).tolist(),
    "price_per_sqft": market_df["PricePerSqFt"].clip(lo_s, hi_s).round(0).tolist(),
}

# ---------- Monthly trend (median close price by month) ----------
market_df["SaleMonth"] = market_df["CloseDate"].dt.to_period("M").astype(str)
monthly_trend = (
    market_df.groupby("SaleMonth")["ClosePrice"]
    .median()
    .reset_index()
    .rename(columns={"ClosePrice": "median_price"})
    .sort_values("SaleMonth")
)

# ---------- Export ----------
joblib.dump(zip_stats, "zip_stats.pkl")
joblib.dump(city_stats, "city_stats.pkl")
joblib.dump(market_summary, "market_summary.pkl")
joblib.dump(price_distribution, "price_distribution.pkl")
joblib.dump(monthly_trend, "monthly_trend.pkl")

print(f"ZIP stats: {len(zip_stats):,} zips")
print(f"City stats: {len(city_stats):,} cities")
print(f"Price distribution points: {len(price_distribution['close_price']):,}")
print(f"Monthly trend rows: {len(monthly_trend):,}")
print("All market data files exported successfully.")

ZIP stats: 1,019 zips
City stats: 927 cities
Price distribution points: 71,057
Monthly trend rows: 7
All market data files exported successfully.
